In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# Install Ultralytics (only once per session)
!pip install ultralytics --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 29.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.1 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 75.8 MB/s eta 0:00:00:00:0100:01


In [2]:
# Cell 0: Environment Setup
import os

os.environ['ULTRALYTICS_ANALYTICS'] = 'False'

In [3]:
# Cell 1: Universal Setup

from ultralytics import YOLO
from ultralytics.utils import SETTINGS  # Import the settings
import yaml
import os

# This line disables the Ray Tune integration that is causing the hang
SETTINGS.update({'raytune': False})

def create_yaml_config(yaml_path, dataset_path, class_names):
    data = {'nc': len(class_names), 'names': class_names}
    for split in ['train', 'val', 'test', 'newtest']:
        split_dir = os.path.join(dataset_path, split)
        if os.path.isdir(split_dir):
            data[split] = os.path.join(split_dir, 'images')
    with open(yaml_path, 'w') as f:
        yaml.dump(data, f, sort_keys=False)

def print_metrics(metrics_obj, model_name, dataset_name, split_name):
    print(f"\n--- {model_name} on {dataset_name} ({split_name} set) ---")
    if not metrics_obj: return
    for k, v in metrics_obj.results_dict.items():
        key = k.replace('metrics/', '').replace('(B)', '').replace('mAP', 'mAP@')
        print(f"{key.capitalize():<12}: {v}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Experiment 1: Medium Model on FSOD

In [5]:
# Cell 2: Train Medium Model on FSOD

dataset_path_fsod = "/kaggle/input/object-detection-lydia/FSOD/FSOD"
class_names_fsod = ['laptop', 'umbrella', 'mobile_phone', 'metal_cup', 'portable_charger_1', 'battery', 'portable_charger_2', 'spray_alcohol', 'iron_shoe', 'glass_bottle', 'lighter', 'metal_can', 'pressure_tank', 'drink_bottle', 'nail_clippers', 'scissor', 'straight_knife', 'multi-tool_knife', 'folding_knife', 'utility_knife']
model_type_m, epochs_m = 'yolov10m', 60
run_name_m_fsod = f'{model_type_m}_fsod_{epochs_m}e'
yaml_path_m_fsod = f'/kaggle/working/{run_name_m_fsod}_config.yaml'
create_yaml_config(yaml_path_m_fsod, dataset_path_fsod, class_names_fsod)

model_m_fsod = YOLO(f'{model_type_m}.pt')
results_m_fsod = model_m_fsod.train(data=yaml_path_m_fsod, epochs=epochs_m, imgsz=640, batch=16, project='fsod_vs_sixray', name=run_name_m_fsod)

100%|██████████| 32.1M/32.1M [00:00<00:00, 173MB/s]


Ultralytics 8.3.154 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolov10m_fsod_60e_config.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov10m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov10m_fsod_60e, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plo

100%|██████████| 755k/755k [00:00<00:00, 17.4MB/s]

Overriding model.yaml nc=80 with nc=20

                   from  n    params  module                                       arguments                     
  0                  -1  1      1392  ultralytics.nn.modules.conv.Conv             [3, 48, 3, 2]                 
  1                  -1  1     41664  ultralytics.nn.modules.conv.Conv             [48, 96, 3, 2]                
  2                  -1  2    111360  ultralytics.nn.modules.block.C2f             [96, 96, 2, True]             
  3                  -1  1    166272  ultralytics.nn.modules.conv.Conv             [96, 192, 3, 2]               
  4                  -1  4    813312  ultralytics.nn.modules.block.C2f             [192, 192, 4, True]           
  5                  -1  1     78720  ultralytics.nn.modules.block.SCDown          [192, 384, 3, 2]              
  6                  -1  4   3248640  ultralytics.nn.modules.block.C2f             [384, 384, 4, True]           
  7                  -1  1    228672  ultralytic

 13                  -1  2   1993728  ultralytics.nn.modules.block.C2f             [960, 384, 2]                 
 14                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']          
 15             [-1, 4]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 16                  -1  2    517632  ultralytics.nn.modules.block.C2f             [576, 192, 2]                 
 17                  -1  1    332160  ultralytics.nn.modules.conv.Conv             [192, 192, 3, 2]              
 18            [-1, 13]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 19                  -1  2    831744  ultralytics.nn.modules.block.C2fCIB          [576, 384, 2, True]           
 20                  -1  1    152448  ultralytics.nn.modules.block.SCDown          [384, 384, 3, 2]              
 21            [-1, 10]  1         0  ultralytics.nn.modules.conv.Concat           [1]  

...

                   all       1233       4070      0.796       0.68      0.761      0.539

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      60/60      7.92G      1.661     0.9022      2.036         27        640: 100%|██████████| 617/617 [06:20<00:00,  1.62it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 39/39 [00:16<00:00,  2.42it/s]


                   all       1233       4070      0.797      0.683       0.76      0.539

60 epochs completed in 6.692 hours.
Optimizer stripped from fsod_vs_sixray/yolov10m_fsod_60e/weights/last.pt, 33.5MB
Optimizer stripped from fsod_vs_sixray/yolov10m_fsod_60e/weights/best.pt, 33.5MB

Validating fsod_vs_sixray/yolov10m_fsod_60e/weights/best.pt...
Ultralytics 8.3.154 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
YOLOv10m summary (fused): 136 layers, 15,324,748 parameters, 0 gradients, 58.9 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 39/39 [00:15<00:00,  2.53it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all       1233       4070      0.821      0.678      0.761      0.541
                laptop        217        269      0.895       0.97      0.977      0.868
              umbrella        179        219      0.879      0.863      0.911      0.642
          mobile_phone        337        540      0.884      0.846      0.922      0.753
             metal_cup        228        270      0.953      0.922      0.976      0.835
    portable_charger_1        149        177      0.858      0.814      0.914      0.659
               battery        319        480      0.782      0.575      0.681      0.394
    portable_charger_2        144        199       0.84      0.789      0.877      0.643
         spray_alcohol        168        249      0.826      0.466      0.589      0.372
             iron_shoe         80        166      0.849      0.614       0.73      0.438
          glass_bottle        174        298      0.793       0.68      0.757      0.486
               lighte

In [8]:
# Cell 3: Validate Medium Model on FSOD

val_metrics = model_m_fsod.val(data=yaml_path_m_fsod, split='val', verbose=False, workers=0)
print_metrics(val_metrics, model_type_m, 'FSOD', 'val')

newtest_metrics = model_m_fsod.val(data=yaml_path_m_fsod, split='newtest', verbose=False, workers=0)
print_metrics(newtest_metrics, model_type_m, 'FSOD', 'newtest')

Ultralytics 8.3.154 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 144.1±64.1 MB/s, size: 215.0 KB)


val: Scanning /kaggle/input/object-detection-lydia/FSOD/FSOD/val/labels... 1233 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1233/1233 [00:01<00:00, 695.02it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/object-detection-lydia/FSOD/FSOD/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 78/78 [00:35<00:00,  2.17it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all       1233       4070      0.817       0.68      0.762      0.541
Speed: 0.2ms preprocess, 18.1ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to fsod_vs_sixray/yolov10m_fsod_60e4

--- yolov10m on FSOD (val set) ---
Precision   : 0.8173982613228554
Recall      : 0.6798980652424037
Map@50      : 0.7616409142957404
Map@50-95   : 0.5413857969811097
Fitness     : 0.5634113087125727
Ultralytics 8.3.154 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 142.5±52.9 MB/s, size: 166.0 KB)


val: Scanning /kaggle/input/object-detection-lydia/FSOD/FSOD/newtest/labels... 1232 images, 1 backgrounds, 0 corrupt: 100%|██████████| 1233/1233 [00:02<00:00, 435.30it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/object-detection-lydia/FSOD/FSOD/newtest is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 78/78 [00:36<00:00,  2.16it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all       1233       3924      0.795      0.703       0.76      0.541
Speed: 0.2ms preprocess, 18.3ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to fsod_vs_sixray/yolov10m_fsod_60e5

--- yolov10m on FSOD (newtest set) ---
Precision   : 0.795336468998291
Recall      : 0.7026722794959581
Map@50      : 0.760274722497208
Map@50-95   : 0.5408754408526921
Fitness     : 0.5628153690171437


Experiment 2: Large Model on FSOD

In [5]:
# Cell 4: Train Large Model on FSOD

dataset_path_fsod = "/kaggle/input/object-detection-lydia/FSOD/FSOD"
class_names_fsod = ['laptop', 'umbrella', 'mobile_phone', 'metal_cup', 'portable_charger_1', 'battery', 'portable_charger_2', 'spray_alcohol', 'iron_shoe', 'glass_bottle', 'lighter', 'metal_can', 'pressure_tank', 'drink_bottle', 'nail_clippers', 'scissor', 'straight_knife', 'multi-tool_knife', 'folding_knife', 'utility_knife']
model_type_l, epochs_l = 'yolov10l', 60

run_name_l_fsod = f'{model_type_l}_fsod_{epochs_l}e'
yaml_path_l_fsod = f'/kaggle/working/{run_name_l_fsod}_config.yaml'
create_yaml_config(yaml_path_l_fsod, dataset_path_fsod, class_names_fsod)

model_l_fsod = YOLO(f'{model_type_l}.pt')
results_l_fsod = model_l_fsod.train(data=yaml_path_l_fsod, epochs=epochs_l, imgsz=640, batch=8, project='fsod_vs_sixray', name=run_name_l_fsod)

Ultralytics 8.3.154 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolov10l_fsod_60e_config.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov10l.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov10l_fsod_60e2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plo

100%|██████████| 755k/755k [00:00<00:00, 16.6MB/s]

Overriding model.yaml nc=80 with nc=20

                   from  n    params  module                                       arguments                     
  0                  -1  1      1856  ultralytics.nn.modules.conv.Conv             [3, 64, 3, 2]                 
  1                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  2                  -1  3    279808  ultralytics.nn.modules.block.C2f             [128, 128, 3, True]           
  3                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  4                  -1  6   2101248  ultralytics.nn.modules.block.C2f             [256, 256, 6, True]           
  5                  -1  1    137728  ultralytics.nn.modules.block.SCDown          [256, 512, 3, 2]              
  6                  -1  6   8396800  ultralytics.nn.modules.block.C2f             [512, 512, 6, True]           
  7                  -1  1    268800  ultralytic

  8                  -1  3   1744384  ultralytics.nn.modules.block.C2fCIB          [512, 512, 3, True]           
  9                  -1  1    656896  ultralytics.nn.modules.block.SPPF            [512, 512, 5]                 
 10                  -1  1    990976  ultralytics.nn.modules.block.PSA             [512, 512]                    
 11                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']          
 12             [-1, 6]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 13                  -1  3   2006528  ultralytics.nn.modules.block.C2fCIB          [1024, 512, 3, True]          
 14                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']          
 15             [-1, 4]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 16                  -1  3   1247744  ultralytics.nn.modules.block.C2f             [768,

100%|██████████| 5.35M/5.35M [00:00<00:00, 71.3MB/s]


...

      60/60      6.83G      1.602     0.8509       2.08          8        640: 100%|██████████| 1234/1234 [09:31<00:00,  2.16it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 78/78 [00:23<00:00,  3.33it/s]

                   all       1233       4070      0.799      0.708      0.771      0.556



60 epochs completed in 9.960 hours.
Optimizer stripped from fsod_vs_sixray/yolov10l_fsod_60e2/weights/last.pt, 52.2MB
Optimizer stripped from fsod_vs_sixray/yolov10l_fsod_60e2/weights/best.pt, 52.2MB

Validating fsod_vs_sixray/yolov10l_fsod_60e2/weights/best.pt...
Ultralytics 8.3.154 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
YOLOv10l summary (fused): 174 layers, 24,324,748 parameters, 0 gradients, 120.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 78/78 [00:20<00:00,  3.72it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all       1233       4070      0.805      0.712      0.778      0.559
                laptop        217        269      0.892      0.957      0.967       0.86
              umbrella        179        219      0.851      0.845      0.888      0.643
          mobile_phone        337        540      0.882       0.87      0.934      0.764
             metal_cup        228        270      0.918      0.911      0.967      0.824
    portable_charger_1        149        177      0.826      0.853      0.926      0.697
               battery        319        480      0.731      0.623      0.705      0.412
    portable_charger_2        144        199      0.849      0.809      0.896      0.665
         spray_alcohol        168        249      0.753      0.486      0.603      0.384
             iron_shoe         80        166      0.838      0.693      0.766      0.482
          glass_bottle        174        298      0.778      0.658      0.743      0.482
               lighte

In [6]:
# Cell 5: Validate Large Model on FSOD

val_metrics = model_l_fsod.val(data=yaml_path_l_fsod, split='val', verbose=False, workers=0)
print_metrics(val_metrics, model_type_l, 'FSOD', 'val')

newtest_metrics = model_l_fsod.val(data=yaml_path_l_fsod, split='newtest', verbose=False, workers=0)
print_metrics(newtest_metrics, model_type_l, 'FSOD', 'newtest')

Ultralytics 8.3.154 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
YOLOv10l summary (fused): 174 layers, 24,324,748 parameters, 0 gradients, 120.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 177.4±66.5 MB/s, size: 215.0 KB)


val: Scanning /kaggle/input/object-detection-lydia/FSOD/FSOD/val/labels... 1233 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1233/1233 [00:02<00:00, 472.32it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/object-detection-lydia/FSOD/FSOD/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 155/155 [00:56<00:00,  2.75it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all       1233       4070      0.809      0.709      0.778      0.559
Speed: 0.2ms preprocess, 32.6ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to fsod_vs_sixray/yolov10l_fsod_60e22

--- yolov10l on FSOD (val set) ---
Precision   : 0.8093818964470787
Recall      : 0.7085821510574756
Map@50      : 0.7778787294956788
Map@50-95   : 0.559154859355524
Fitness     : 0.5810272463695395
Ultralytics 8.3.154 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 1.1±0.8 ms, read: 14.2±4.5 MB/s, size: 166.0 KB)


val: Scanning /kaggle/input/object-detection-lydia/FSOD/FSOD/newtest/labels... 1232 images, 1 backgrounds, 0 corrupt: 100%|██████████| 1233/1233 [00:15<00:00, 79.58it/s]


WARNING ⚠️ val: Cache directory /kaggle/input/object-detection-lydia/FSOD/FSOD/newtest is not writeable, cache not saved.


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 155/155 [01:01<00:00,  2.50it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all       1233       3924      0.818      0.702       0.77      0.555
Speed: 0.2ms preprocess, 32.7ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to fsod_vs_sixray/yolov10l_fsod_60e23

--- yolov10l on FSOD (newtest set) ---
Precision   : 0.8184049522789252
Recall      : 0.7016859147726473
Map@50      : 0.7698422421505947
Map@50-95   : 0.5551239835911754
Fitness     : 0.5765958094471174


Experiment 3: Medium Model on SIXray

In [18]:
# Cell 6: Train Medium Model on SIXray

dataset_path_sixray = "/kaggle/input/object-detection-lydia/SIXray/SIXray"
class_names_sixray = ['Gun', 'Knife', 'Pliers', 'Scissors', 'Wrench']
model_type_m, epochs_m_sixray = 'yolov10m', 1
run_name_m_sixray = f'{model_type_m}_sixray_{epochs_m_sixray}e'
yaml_path_m_sixray = f'/kaggle/working/{run_name_m_sixray}_config.yaml'
create_yaml_config(yaml_path_m_sixray, dataset_path_sixray, class_names_sixray)

model_m_sixray = YOLO(f'{model_type_m}.pt')
results_m_sixray = model_m_sixray.train(data=yaml_path_m_sixray, epochs=epochs_m_sixray, imgsz=640, batch=16, project='fsod_vs_sixray', name=run_name_m_sixray)

Ultralytics 8.3.154 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolov10m_sixray_1e_config.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov10m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov10m_sixray_1e, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, pl

train: Scanning /kaggle/input/object-detection-lydia/SIXray/SIXray/train/labels... 5819 images, 13 backgrounds, 0 corrupt: 100%|██████████| 5819/5819 [00:24<00:00, 241.55it/s]


WARNING ⚠️ train: Cache directory /kaggle/input/object-detection-lydia/SIXray/SIXray/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 9.2±2.4 MB/s, size: 42.3 KB)


val: Scanning /kaggle/input/object-detection-lydia/SIXray/SIXray/val/labels... 1662 images, 2 backgrounds, 0 corrupt: 100%|██████████| 1662/1662 [00:07<00:00, 227.61it/s]


WARNING ⚠️ val: Cache directory /kaggle/input/object-detection-lydia/SIXray/SIXray/val is not writeable, cache not saved.
Plotting labels to fsod_vs_sixray/yolov10m_sixray_1e/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.001111, momentum=0.9) with parameter groups 129 weight(decay=0.0), 142 weight(decay=0.0005), 141 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to fsod_vs_sixray/yolov10m_sixray_1e
Starting training for 1 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


        1/1      8.79G      3.654      5.667      3.692         22        640: 100%|██████████| 364/364 [03:50<00:00,  1.58it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 52/52 [00:22<00:00,  2.28it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all       1662       3176      0.245      0.359       0.24     0.0961

1 epochs completed in 0.072 hours.
Optimizer stripped from fsod_vs_sixray/yolov10m_sixray_1e/weights/last.pt, 33.5MB
Optimizer stripped from fsod_vs_sixray/yolov10m_sixray_1e/weights/best.pt, 33.5MB

Validating fsod_vs_sixray/yolov10m_sixray_1e/weights/best.pt...
Ultralytics 8.3.154 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
YOLOv10m summary (fused): 136 layers, 15,316,063 parameters, 0 gradients, 58.9 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 52/52 [00:19<00:00,  2.71it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all       1662       3176      0.246      0.362       0.24     0.0963
                   Gun        551        888       0.24      0.671      0.428      0.177
                 Knife        291        442      0.188       0.33      0.138     0.0438
                Pliers        794       1110      0.488       0.14      0.229     0.0947
              Scissors        178        206      0.129      0.553      0.325      0.133
                Wrench        392        530      0.185      0.116     0.0822     0.0325
Speed: 0.1ms preprocess, 8.6ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to fsod_vs_sixray/yolov10m_sixray_1e


In [19]:
# Cell 7: Validate Medium Model on SIXray

val_metrics = model_m_sixray.val(data=yaml_path_m_sixray, split='val', verbose=False, workers=0)
print_metrics(val_metrics, model_type_m, 'SIXray', 'val')

test_metrics = model_m_sixray.val(data=yaml_path_m_sixray, split='test', verbose=False, workers=0)
print_metrics(test_metrics, model_type_m, 'SIXray', 'test')

Ultralytics 8.3.154 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
YOLOv10m summary (fused): 136 layers, 15,316,063 parameters, 0 gradients, 58.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 88.4±49.4 MB/s, size: 41.0 KB)


val: Scanning /kaggle/input/object-detection-lydia/SIXray/SIXray/val/labels... 1662 images, 2 backgrounds, 0 corrupt: 100%|██████████| 1662/1662 [00:01<00:00, 888.48it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/object-detection-lydia/SIXray/SIXray/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 104/104 [00:40<00:00,  2.55it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all       1662       3176      0.244      0.362      0.239     0.0959
Speed: 0.2ms preprocess, 18.4ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to fsod_vs_sixray/yolov10m_sixray_1e2

--- yolov10m on SIXray (val set) ---
Precision   : 0.2441788504153614
Recall      : 0.3620331166110936
Map@50      : 0.23947063109686778
Map@50-95   : 0.09588146493547932
Fitness     : 0.11024038155161817
Ultralytics 8.3.154 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 8.4±3.6 MB/s, size: 46.7 KB)


val: Scanning /kaggle/input/object-detection-lydia/SIXray/SIXray/test/labels... 831 images, 2 backgrounds, 0 corrupt: 100%|██████████| 831/831 [00:04<00:00, 207.71it/s]


WARNING ⚠️ val: Cache directory /kaggle/input/object-detection-lydia/SIXray/SIXray/test is not writeable, cache not saved.


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 52/52 [00:20<00:00,  2.50it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        831       1593      0.281      0.324      0.239     0.0983
Speed: 0.2ms preprocess, 18.0ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to fsod_vs_sixray/yolov10m_sixray_1e3

--- yolov10m on SIXray (test set) ---
Precision   : 0.28086915994950895
Recall      : 0.32416512407843023
Map@50      : 0.23899258464083423
Map@50-95   : 0.09832192226314614
Fitness     : 0.11238898850091494


Experiment 4: Large Model on SIXray

In [20]:
# Cell 8: Train Large Model on SIXray

dataset_path_sixray = "/kaggle/input/object-detection-lydia/SIXray/SIXray"
class_names_sixray = ['Gun', 'Knife', 'Pliers', 'Scissors', 'Wrench']
model_type_l, epochs_l_sixray = 'yolov10l', 1

run_name_l_sixray = f'{model_type_l}_sixray_{epochs_l_sixray}e'
yaml_path_l_sixray = f'/kaggle/working/{run_name_l_sixray}_config.yaml'
create_yaml_config(yaml_path_l_sixray, dataset_path_sixray, class_names_sixray)

model_l_sixray = YOLO(f'{model_type_l}.pt')
results_l_sixray = model_l_sixray.train(data=yaml_path_l_sixray, epochs=epochs_l_sixray, imgsz=640, batch=8, project='fsod_vs_sixray', name=run_name_l_sixray)

Ultralytics 8.3.154 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolov10l_sixray_1e_config.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov10l.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov10l_sixray_1e, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plo

train: Scanning /kaggle/input/object-detection-lydia/SIXray/SIXray/train/labels... 5819 images, 13 backgrounds, 0 corrupt: 100%|██████████| 5819/5819 [00:07<00:00, 824.66it/s]


WARNING ⚠️ train: Cache directory /kaggle/input/object-detection-lydia/SIXray/SIXray/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 78.0±21.6 MB/s, size: 42.3 KB)


val: Scanning /kaggle/input/object-detection-lydia/SIXray/SIXray/val/labels... 1662 images, 2 backgrounds, 0 corrupt: 100%|██████████| 1662/1662 [00:01<00:00, 1029.36it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/object-detection-lydia/SIXray/SIXray/val is not writeable, cache not saved.


Plotting labels to fsod_vs_sixray/yolov10l_sixray_1e/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.001111, momentum=0.9) with parameter groups 167 weight(decay=0.0), 180 weight(decay=0.0005), 179 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to fsod_vs_sixray/yolov10l_sixray_1e
Starting training for 1 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


        1/1      7.55G      3.722      5.682      4.007          5        640: 100%|██████████| 728/728 [05:48<00:00,  2.09it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 104/104 [00:32<00:00,  3.19it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all       1662       3176      0.479      0.265      0.231     0.0941

1 epochs completed in 0.107 hours.
Optimizer stripped from fsod_vs_sixray/yolov10l_sixray_1e/weights/last.pt, 52.2MB
Optimizer stripped from fsod_vs_sixray/yolov10l_sixray_1e/weights/best.pt, 52.2MB

Validating fsod_vs_sixray/yolov10l_sixray_1e/weights/best.pt...
Ultralytics 8.3.154 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
YOLOv10l summary (fused): 174 layers, 24,313,183 parameters, 0 gradients, 120.0 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 104/104 [00:27<00:00,  3.74it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all       1662       3176       0.48      0.264      0.231     0.0941
                   Gun        551        888      0.459      0.503       0.48      0.197
                 Knife        291        442      0.289      0.262      0.174     0.0727
                Pliers        794       1110      0.289      0.463      0.303      0.124
              Scissors        178        206      0.361     0.0922      0.131     0.0542
                Wrench        392        530          1          0     0.0676     0.0233
Speed: 0.2ms preprocess, 13.8ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to fsod_vs_sixray/yolov10l_sixray_1e


In [21]:
# Cell 9: Validate Large Model on SIXray

val_metrics = model_l_sixray.val(data=yaml_path_l_sixray, split='val', verbose=False, workers=0)
print_metrics(val_metrics, model_type_l, 'SIXray', 'val')

test_metrics = model_l_sixray.val(data=yaml_path_l_sixray, split='test', verbose=False, workers=0)
print_metrics(test_metrics, model_type_l, 'SIXray', 'test')

Ultralytics 8.3.154 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
YOLOv10l summary (fused): 174 layers, 24,313,183 parameters, 0 gradients, 120.0 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 96.4±57.7 MB/s, size: 41.0 KB)


val: Scanning /kaggle/input/object-detection-lydia/SIXray/SIXray/val/labels... 1662 images, 2 backgrounds, 0 corrupt: 100%|██████████| 1662/1662 [00:02<00:00, 798.98it/s]


WARNING ⚠️ val: Cache directory /kaggle/input/object-detection-lydia/SIXray/SIXray/val is not writeable, cache not saved.


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 208/208 [01:05<00:00,  3.16it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all       1662       3176       0.48      0.265      0.231     0.0942
Speed: 0.2ms preprocess, 33.5ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to fsod_vs_sixray/yolov10l_sixray_1e2

--- yolov10l on SIXray (val set) ---
Precision   : 0.48044867517489614
Recall      : 0.26476819502804655
Map@50      : 0.23113073495642658
Map@50-95   : 0.0942315647405585
Fitness     : 0.1079214817621453
Ultralytics 8.3.154 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 103.3±31.6 MB/s, size: 46.7 KB)


val: Scanning /kaggle/input/object-detection-lydia/SIXray/SIXray/test/labels... 831 images, 2 backgrounds, 0 corrupt: 100%|██████████| 831/831 [00:01<00:00, 716.64it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/object-detection-lydia/SIXray/SIXray/test is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 104/104 [00:32<00:00,  3.17it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        831       1593      0.423      0.275      0.228     0.0925
Speed: 0.2ms preprocess, 33.4ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to fsod_vs_sixray/yolov10l_sixray_1e3

--- yolov10l on SIXray (test set) ---
Precision   : 0.42308176468239234
Recall      : 0.27481324577162536
Map@50      : 0.227846472993286
Map@50-95   : 0.09252138259043109
Fitness     : 0.10605389163071657
